# Lab 2 Part 1: RAG runtime

The first code cell creates a pinned environment and prints its commit, versions,
device and seed. It does not replace packages inside the running notebook kernel.
Windows, Linux, Colab and Kaggle paths are handled without a hard-coded `/content`
directory on a PC. First-time setup needs internet and several GB of free disk.

Use a fresh kernel and Run All. The default is a full run. `LAB2_SMOKE_ONLY=1`
is only for developer checks and is clearly printed; those outputs are not a
completed GPU experiment. Keep your real executed copy after the full run.
Run only one GPU training/generation notebook at a time. Setup is locked, but
two concurrent large models can still exceed the GPU's memory.

This remains the repository's extractive retrieval baseline, not a claim that all graded RAG features are finished.

In [1]:
from pathlib import Path
import os, subprocess, sys, platform, json, time

os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "60")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

SMOKE_ONLY = os.environ.get("LAB2_SMOKE_ONLY", "0") == "1"
roots = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in roots if (p / "tools/setup_runtime.py").exists()), None)
if ROOT is None:
    if os.environ.get("LAB2_ROOT"):
        ROOT = Path(os.environ["LAB2_ROOT"]).expanduser().resolve()
    elif Path("/content").exists():
        ROOT = Path("/content/Lab-2")
    elif Path("/kaggle/working").exists():
        ROOT = Path("/kaggle/working/Lab-2")
    else:
        ROOT = Path.home() / "DATA266" / "Lab-2"
    if not ROOT.exists():
        ROOT.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", "https://github.com/denishatank12/Lab-2.git", str(ROOT)], check=True)
    if not (ROOT / "tools/setup_runtime.py").exists():
        raise RuntimeError(f"{ROOT} is an older/incomplete checkout. Update it with git pull, then restart this notebook.")
print("Notebook host:", platform.python_version(), "repository:", ROOT)
print("Mode:", "SMOKE ONLY: no full training or model-backed evaluation" if SMOKE_ONLY else "FULL RUN")
# Refresh a clean main checkout, but never overwrite local edits or reset a branch.
if not SMOKE_ONLY and os.environ.get("LAB2_UPDATE_REPOSITORY", "1") == "1":
    branch = subprocess.check_output(["git", "-C", str(ROOT), "branch", "--show-current"], text=True).strip()
    dirty = subprocess.check_output(["git", "-C", str(ROOT), "status", "--porcelain"], text=True).strip()
    if branch == "main" and not dirty:
        subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
    else:
        print("Kept existing checkout: local edits or a non-main branch. No files were reset.")
if "SETUP_SCHEMA = 2" not in (ROOT / "tools/setup_runtime.py").read_text(encoding="utf-8"):
    # A freshly downloaded notebook must not silently execute the old broken installer.
    previous_root = ROOT
    ROOT = ROOT.with_name(ROOT.name + "-notebook-v2")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "https://github.com/denishatank12/Lab-2.git", str(ROOT)], check=True)
    if not (ROOT / "tools/setup_runtime.py").is_file() or "SETUP_SCHEMA = 2" not in (ROOT / "tools/setup_runtime.py").read_text(encoding="utf-8"):
        raise RuntimeError(f"Update the checkout at {ROOT} before running. The original checkout was preserved.")
    print("Using repaired code at", ROOT, "; the older checkout and its results remain at", previous_root)
subprocess.run([sys.executable, str(ROOT / "tools/setup_runtime.py"),
                "--device", "cpu" if SMOKE_ONLY else os.environ.get("LAB2_DEVICE", "auto")], check=True)
requested_device = "cpu" if SMOKE_ONLY else os.environ.get("LAB2_DEVICE", "auto")
import shutil
use_cuda = requested_device == "cuda" or (requested_device == "auto" and shutil.which("nvidia-smi") is not None)
runtime_file = ROOT / (".lab2_runtime_cuda.json" if use_cuda else ".lab2_runtime_cpu.json")
PYTHON = Path(json.loads(runtime_file.read_text())["python"])
SEED = 266
print("All remaining commands use:", PYTHON)

def run_script(relative, *arguments):
    command = [str(PYTHON), "-u", str(ROOT / relative), *map(str, arguments)]
    logs = ROOT / "runtime_checks"
    logs.mkdir(exist_ok=True)
    log = logs / (Path(relative).stem + ".log")
    print("Running", Path(relative).name, flush=True)
    with log.open("a", encoding="utf-8") as handle:
        process = subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
        for line in process.stdout:
            print(line, end="", flush=True)
            handle.write(line)
            handle.flush()
        code = process.wait()
    if code:
        raise RuntimeError(f"{Path(relative).name} stopped (exit {code}). Full diagnostic log: {log}")


Notebook host: 3.12.6 repository: /Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2
Mode: FULL RUN
Kept existing checkout: local edits or a non-main branch. No files were reset.


{
  "commit": "3b12f56",
  "working_tree_dirty": true,
  "python": "3.12.6",
  "seed": 266,
  "executable": "/Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/.venv/bin/python",
  "versions": {
    "torch": "2.5.1",
    "torchvision": "0.20.1",
    "diffusers": "0.31.0",
    "transformers": "4.46.2",
    "peft": "0.13.2",
    "datasets": "3.1.0",
    "huggingface-hub": "0.26.2"
  },
  "cuda_available": false,
  "gpu": "cpu"
}
PASS: core imports, matching torchvision kernels and device preflight


LAB2_PYTHON=/Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/.venv/bin/python
All remaining commands use: /Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/.venv/bin/python


## Smoke test
Check PDF ingestion, trusted-page filtering and lexical retrieval before downloading an embedding model.

In [2]:
run_script("part1_rag/src/smoke_test.py")

Running smoke_test.py


PASS: indexed 179 passages; retrieval returned 3 grounded hits


## Generate and validate
The hidden-set questions path can be supplied through `QUESTIONS_PATH`. It is also passed to the unchanged official validator. A schema-valid file alone does not prove answer correctness.

In [3]:
DATA = Path(os.environ.get("LAB2_DATA_DIR", str(ROOT / "part1_rag/data"))).resolve()
QUESTIONS = Path(os.environ.get("QUESTIONS_PATH", str(DATA / "Questions.csv"))).resolve()
OUTPUT = ROOT / "part1_rag/outputs/submission.csv"
run_script("part1_rag/src/generate_submission.py", "--data-dir", DATA,
           "--questions-path", QUESTIONS, "--output", OUTPUT,
           "--lexical-only" if SMOKE_ONLY else "--require-dense")
run_script("part1_rag/data/validate_submission.py", OUTPUT, QUESTIONS)
import csv
with OUTPUT.open(encoding="utf-8", newline="") as handle:
    rows = list(csv.DictReader(handle))
print("Validated rows:", len(rows), "file:", OUTPUT)
print(json.dumps(rows[:2], indent=2))

Running generate_submission.py


wrote 40 rows to /Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/part1_rag/outputs/submission.csv


Running validate_submission.py


valid: 40 rows


Validated rows: 40 file: /Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/part1_rag/outputs/submission.csv
[
  {
    "id": "1",
    "conversation_id": "C001",
    "question_id": "Q001",
    "answer": "economic crisis in the near future. In short, while the worst of the crisis is probably over, it has left many problems in its wake, which will keep macroeconomists and policy makers busy for many years to come. We shall return to these issues in more detail at many points in the book. In the rest of the chapter, we take a closer look at the three main economic powers of the world: the United States, the Euro area, and China. 1-2 The United States When economists first look at a country, the first two questions they ask are: How big is the country, from an economic point of view? And what is its standard of living? To answer the first, they l economies have turned around. As you can see from Table 1-1, Figure 1-1 Stock prices in the United States, the Euro area, and emerging economies, 2